# Person 1 – Datarensing (Even)

Denne notebooken leser inn rådataene, renser dem og lager de felles filene som resten av gruppen bruker:

- `data/zomato_india_clean.csv` – hele det rensede datasettet
- `data/train.csv` og `data/test.csv` – felles 80/20-splitt (`random_state=42`, stratifisert på `high_rating`)

Kjør cellene ovenfra og ned. Notebooken ligger i `Even/` og leser rådataene fra roten av repoet.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW = Path("../Zomato Restaurant Dataset.csv")
OUT = Path("../data")
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42  # felles i gruppen

## 1. Les inn og se på rådataene
*Skriv med egne ord: hva er en rad, og hvilke kolonner er relevante for problemstillingen?*
Hver rad er assosiert med én resursnt. De kolonnene vi er intresert i er rating, pris, antall anmeldeser, lokasjon(og valuta), og om resturantent tilbyr bordbestilling og levering. Antall stemmer forteller oss noe om popularitet (men fortsatt sårbar for ny åpnede resturanter). Hvilke land og valuta som brukes for filtrering, mens by brukes som forklaringsvariabel. Tjenester tilbudt brukes også som forklaringsvariabler.

In [2]:
raw = pd.read_csv(RAW, encoding="utf-8-sig")
print(raw.shape)
raw.head()

(9551, 21)


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

## 2. Hva må ryddes?
Fire ting å undersøke før vi filtrerer:
1. Hvilke land og valutaer finnes? (Pris i ulike valutaer kan ikke sammenlignes.)
2. Hva betyr vurdering 0?
3. Finnes det umulige verdier, som pris 0?
4. Er noen kolonner nesten konstante?

In [4]:
# 1. Land og valuta
print(raw["Country Code"].value_counts().head(), "\n")
print(raw.groupby("Country Code")["Currency"].first().head())

Country Code
1      8652
216     434
215      80
30       60
189      60
Name: count, dtype: int64 

Country Code
1         Indian Rupees(Rs.)
14                 Dollar($)
30        Brazilian Real(R$)
37                 Dollar($)
94    Indonesian Rupiah(IDR)
Name: Currency, dtype: object


In [5]:
# 2. Vurdering 0: hvilken ratingtekst har de?
print(raw.loc[raw["Aggregate rating"] == 0, "Rating text"].value_counts())
print("Laveste vurdering over 0:", raw.loc[raw["Aggregate rating"] > 0, "Aggregate rating"].min())

Rating text
Not rated    2148
Name: count, dtype: int64
Laveste vurdering over 0: 1.8


In [6]:
# 3. Pris 0 og 4. nesten konstante kolonner
print("Pris = 0:", (raw["Average Cost for two"] == 0).sum())
for col in ["Switch to order menu", "Is delivering now", "Has Table booking", "Has Online delivery"]:
    print(col, raw[col].value_counts().to_dict())

Pris = 0: 18
Switch to order menu {'No': 9551}
Is delivering now {'No': 9517, 'Yes': 34}
Has Table booking {'No': 8393, 'Yes': 1158}
Has Online delivery {'No': 7100, 'Yes': 2451}


*Skriv med egne ord: hva fant du, og hva betyr det for rensingen?*
   Hvilke land og valutaer finnes? (Pris i ulike valutaer kan ikke sammenlignes.)
   Hva betyr vurdering 0?
   Finnes det umulige verdier, som pris 0?
   Er noen kolonner nesten konstante?

1. Det finnes flere ulike valutaer. Det vanligste er rupee. Vi fliterer derfor bort alt annet. Ulike valutaer er krevende å sammenligne. 
2. Vurdering 0 er en del av rådataen vår. Det er 2148 resturanter oppført som "Not rated". Disse fjernes fra dataen og vi får vår laveste rating på 1.8. "Not rated" kan være nye steder, stengte steder eller bare tapt data.
3. Det finnes 18 eksempler på pris = 0. Det gir ikke mening fordi ingen resturant er gratis så vi fjerner disse fra datasettet. 
4. Det blir et par konstante kolonner i det rensede datasettet. Land, og valuta blir konstant, men dermed fjernet fra datasett. Switch to order menu" er også "No" for samtlige resturanter så det fjernes. I tilegg har vi "Is deliviering now" dårlig fordi det er tidsavhening. Det gir ikke mening å bruke fordi det handler bare om når dataen ble hentet. "Switch to order menu" 

## 3. Filtrering
Vi logger antall rader etter hvert steg. Tabellen brukes i rapportens dataseksjon.

In [7]:
steps = [("Rådata", len(raw))]

df = raw[raw["Country Code"] == 1]              # bare India: én valuta (rupi)
steps.append(("Bare India", len(df)))

df = df[df["Aggregate rating"] > 0]             # 0 = ikke vurdert, altså manglende verdi
steps.append(("Fjernet ikke-vurderte (vurdering 0)", len(df)))

df = df[df["Average Cost for two"] > 0]         # pris 0 er umulig, og log(0) er udefinert
steps.append(("Fjernet pris 0", len(df)))

row_table = pd.DataFrame(steps, columns=["Steg", "Antall rader"])
row_table

,Steg,Antall rader
0,Rådata,9551
1,Bare India,8652
2,Fjernet ikke-vurderte (vurdering 0),6513
3,Fjernet pris 0,6504


## 4. Nye variabler
- Korte kolonnenavn uten mellomrom (kreves av statsmodels-formler)
- Yes/No → 1/0
- Log-transformasjon av skjeve variabler (pris og stemmer)
- Små byer slått sammen til «Other»
- `high_rating` = 1 hvis vurdering ≥ 3,5 (responsen i klassifikasjonen)

In [8]:
df = df.rename(columns={
    "Aggregate rating": "rating",
    "Average Cost for two": "cost",
    "Votes": "votes",
    "Has Table booking": "table_booking",
    "Has Online delivery": "online_delivery",
    "Price range": "price_range",
    "City": "city",
})

for col in ["table_booking", "online_delivery"]:
    df[col] = (df[col] == "Yes").astype(int)

df["log_cost"] = np.log(df["cost"])
df["log_votes"] = np.log(df["votes"])

df["city"] = df["city"].where(df["city"].isin(["New Delhi", "Gurgaon", "Noida"]), "Other")

df["high_rating"] = (df["rating"] >= 3.5).astype(int)

# Behold bare kolonnene vi bruker. "Rating text"/"Rating color" er vurderingen omkodet
# og ville lekke fasiten inn i modellene, så de tas ikke med.
cols = ["rating", "high_rating", "cost", "log_cost", "votes", "log_votes",
        "price_range", "table_booking", "online_delivery", "city"]
df = df[cols].reset_index(drop=True)
df.head()

,rating,high_rating,cost,log_cost,votes,log_votes,price_range,table_booking,online_delivery,city
0,3.9,1,850,6.745236,140,4.941642,3,0,0,Other
1,3.5,1,700,6.551080,71,4.262680,2,0,0,Other
2,3.6,1,500,6.214608,94,4.543295,2,0,0,Other
3,4.0,1,400,5.991465,87,4.465908,2,0,0,Other
4,4.2,1,1000,6.907755,177,5.176150,3,0,0,Other


## 5. Kontroller

In [9]:
print("Manglende verdier:", df.isna().sum().sum())
print("Duplikater i rådata (Restaurant ID):", raw["Restaurant ID"].duplicated().sum())
print("Andel høyt vurdert:", round(df["high_rating"].mean(), 3))
print("Restauranter med pris > 4000 rupi:", (df["cost"] > 4000).sum())
print(df["city"].value_counts())
df.describe().round(2)

Manglende verdier: 0
Duplikater i rådata (Restaurant ID): 0
Andel høyt vurdert: 0.407
Restauranter med pris > 4000 rupi: 40
city
New Delhi    4048
Gurgaon       890
Other         870
Noida         696
Name: count, dtype: int64


,rating,high_rating,cost,log_cost,votes,log_votes,price_range,table_booking,online_delivery
count,6504.00,6504.00,6504.00,6504.00,6504.00,6504.00,6504.00,6504.00,6504.00
mean,3.35,0.41,716.90,6.30,182.13,3.97,1.88,0.16,0.36
std,0.50,0.49,646.89,0.72,485.38,1.51,0.88,0.37,0.48
min,1.80,0.00,50.00,3.91,4.00,1.39,1.00,0.00,0.00
25%,3.00,0.00,350.00,5.86,16.00,2.77,1.00,0.00,0.00
50%,3.30,0.00,500.00,6.21,49.00,3.89,2.00,0.00,0.00
75%,3.70,1.00,800.00,6.68,149.00,5.00,2.00,0.00,1.00
max,4.90,1.00,8000.00,8.99,10934.00,9.30,4.00,1.00,1.00


In [10]:
# Korrelasjon mellom prisklasse og pris: begrunner at bare én av dem brukes
df[["cost", "log_cost", "price_range"]].corr().round(2)

,cost,log_cost,price_range
cost,1.00,0.86,0.83
log_cost,0.86,1.00,0.89
price_range,0.83,0.89,1.00


*Skriv med egne ord: hvorfor beholder vi de dyre restaurantene (> 4000 rupi), og hvorfor brukes ikke `price_range` sammen med `log_cost`?*
Vi velger å beholde ekstremalveridene fordi det ikke er noe tegn til at det er feil data. Den dyreste resturanten koster 8000 rupee for 2. Det er ikke urimelig dyrt, men en fin luksus resturant. Ved å bruke log_cost tar vi den naturlige logaritmen av prisen. Da får v presset sammen de store verdien slik at de får en sammenlignbart mindre effekt på datasettet. Vi bruker ikke price_range fordi det er mindre presist enn den faktiske prisen. Vi bruker ikke begge fordi da får vi multikolinearitet. Da klarer ikke modellen å skille hvilke variable som har effekten.

## 6. Felles train/test-splitt og lagring

In [11]:
train, test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE,
                               stratify=df["high_rating"])

df.to_csv(OUT / "zomato_india_clean.csv", index=False)
train.to_csv(OUT / "train.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

print(f"Treningssett: {len(train)} rader, andel høyt vurdert {train['high_rating'].mean():.3f}")
print(f"Testsett:     {len(test)} rader, andel høyt vurdert {test['high_rating'].mean():.3f}")

Treningssett: 5203 rader, andel høyt vurdert 0.407
Testsett:     1301 rader, andel høyt vurdert 0.407


## Variabeltabell (til rapporten)

| Variabel | Betydning | Type / enhet |
|---|---|---|
| `rating` | Gjennomsnittlig Zomato-vurdering | Kontinuerlig, 1–5 |
| `high_rating` | 1 hvis `rating` ≥ 3,5 | Binær |
| `cost` / `log_cost` | Gjennomsnittspris for to personer / log av den | Rupi / log(rupi) |
| `votes` / `log_votes` | Antall vurderinger / log av det | Antall / log(antall) |
| `price_range` | Prisklasse | Ordinal, 1–4 (brukes ikke sammen med `log_cost`) |
| `table_booking` | Tilbyr bordbestilling | Binær |
| `online_delivery` | Tilbyr nettlevering | Binær |
| `city` | New Delhi, Gurgaon, Noida eller Other | Kategorisk |

## Bruk i de andre notebookene
```python
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
```